# Clustering the books: K-Means and HDBSCAN

This notebook groups the books into clusters of similar books, using the vectors made by the pipeline
(`python scripts/pipeline.py`). It tries many settings of two clustering methods, picks the best and most
stable setting of each, and describes what every cluster is about.

- **K-Means** is the main model: every book gets a cluster.
- **HDBSCAN** finds dense, very specific clusters (Nordic noir, Venice, ancient Rome...) and leaves the
  rest as "noise". It's kept in two versions: *natural* (noise stays unclustered) and *forced* (every noise
  book is moved to the nearest cluster).

**Input:** `data/processed/clustering_inputs/` (written by `scripts/reduce_umap.py` and `scripts/genre_tags.py`)
**Output:** `data/processed/clustering_results/`: cluster assignments, cluster descriptions, scores, figures
**Run time:** about 1 h 45 min on a laptop (most of it is the HDBSCAN settings).

**How clusters are scored**

| Score | Question it answers | How to read it |
|---|---|---|
| Purity lift | Does each cluster mostly hold one sub-genre (from readers' shelves)? | Purity minus the purity of random clusters of the same sizes. 0 = no better than chance |
| c-TF-IDF lift | Does each cluster have its own vocabulary? | The same idea for words. Falls as the number of clusters grows |
| Premise lift | Are clusters about a *setting or premise* (place, era, sleuth, plot, cozy theme)? | How much more common each cluster's top setting tag is than in random clusters |
| Silhouette | Are clusters tight and well separated? | -1 to 1; only used as a filter here (it favours few, round clusters) |
| Stability | Do we get the same clusters again with another UMAP seed, or on 90% of the books? | Adjusted Rand Index: 1 = identical, 0 = chance |

**How the best setting is picked**
1. Keep only usable results: 15-60 clusters, no cluster above 15% of the books, silhouette at least 0.2.
2. Rank what's left on purity lift, c-TF-IDF lift and premise lift (average percentile rank = `rank_score`).
3. UMAP has some randomness, so the scores move a little from one UMAP seed to the next. Settings whose
   `rank_score` is within that seed noise of the best are a tie: of those, pick the **most stable** one.
4. Run the pick on all UMAP seeds and keep the seed whose clusters agree most with the others.

## 1. Setup

In [ ]:
import ast
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import sparse
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.metrics.cluster import contingency_matrix
from sklearn.preprocessing import normalize

pl.Config.set_tbl_rows(70)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_width_chars(250)

# Paths (works whether the notebook is opened from notebooks/ or the project folder)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed" / "clustering_inputs"
OUTPUT_DIR = ROOT / "data" / "processed" / "clustering_results"

In [ ]:
# ---- Settings ----
SEED = 42

# Which K-Means and HDBSCAN settings to try
KMEANS_K = list(range(15, 51))  # number of clusters
HDBSCAN_MIN_CLUSTER_SIZE = [10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500]
HDBSCAN_MIN_SAMPLES = [1, 5, 10, None]  # None = same as min_cluster_size
HDBSCAN_METHODS = ["eom", "leaf"]

# Rules for a usable result
MIN_CLUSTERS, MAX_CLUSTERS = 15, 60  # a useful number of groups for a recommender
MAX_CLUSTER_SHARE = 0.15             # no single cluster may hold more than 15% of the books
MIN_SILHOUETTE = 0.2                 # clusters must be at least somewhat separated
MAX_NATURAL_NOISE = 0.75             # natural HDBSCAN may leave out many books, but not almost all of them

# How the best setting is picked
HDBSCAN_SHORTLIST = 5   # HDBSCAN: the 5 best settings on the first UMAP seed are re-run on the other seeds
SUBSAMPLES = 2          # K-Means stability: refits on this many random 90% samples of the books

# Scoring details
SILHOUETTE_SAMPLE = 5_000    # silhouette is slow, so it's computed on a sample of books
TOP_N_WORDS = 10             # keywords per cluster
PREMISE_GROUPS = ["place", "era", "sleuth", "plot", "cozy_theme"]  # tag groups about setting/premise
MIN_LIFT, MATCH_COVERAGE = 2.0, 0.4  # a cluster "matches" a tag that covers >= 40% of it and is >= 2x as common as usual
TSNE_MAX_POINTS = 15_000     # the cluster map shows a random sample of this many books

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

## 2. Load the data

`X[seed]` holds the UMAP vectors for each UMAP seed (one row of numbers per book). The first seed is the main one.

In [ ]:
books = pl.read_parquet(DATA_DIR / "books.parquet")
tags = pl.read_parquet(DATA_DIR / "genre_tags.parquet")
assert tags["book_id"].to_list() == books["book_id"].to_list(), "run scripts/genre_tags.py again"

seed_files = sorted(DATA_DIR.glob("embeddings_umap_seed*.npy"))
SEEDS = sorted(int(f.stem.split("seed")[1]) for f in seed_files)
SEEDS = [42] + [s for s in SEEDS if s != 42] if 42 in SEEDS else SEEDS
X = {s: np.load(DATA_DIR / f"embeddings_umap_seed{s}.npy") for s in SEEDS}
N = books.height
assert all(x.shape[0] == N for x in X.values()), "the UMAP files and the book table must have the same rows"
MAIN = SEEDS[0]

genre = tags["primary_genre"].to_numpy()        # each book's most distinctive sub-genre ("general" = none)
counts_for_purity = genre != "general"
titles = books["title"].to_numpy()
docs = books["document"].fill_null("").to_list()  # description + reviews, for keywords
print(f"{N:,} books, {X[MAIN].shape[1]} numbers each, UMAP seeds {SEEDS}")
print(f"Books with a sub-genre tag: {counts_for_purity.mean():.1%}")
tags.group_by("primary_genre").len().sort("len", descending=True).head(15)

## 3. Scoring tools

**c-TF-IDF** finds the words that make each cluster different from the others: it joins all the text in a
cluster into one document, then scores each word by how common it is in that cluster compared with all
clusters. The word counts for every book are computed once here, so scoring each clustering is fast.

**Sub-genre tags** come from the shelves readers put each book on (`genre_tags.py`). They are never used to
build the clusters, only to check them.

In [ ]:
# Words that appear in almost every book review, so they say nothing about a cluster.
EXTRA_STOP_WORDS = {
    "book", "books", "read", "reading", "reader", "readers", "story", "stories", "novel", "novels",
    "author", "authors", "writing", "written", "character", "characters", "series", "page", "pages",
    "chapter", "chapters", "plot", "really", "just", "like", "good", "great", "loved", "love",
    "enjoyed", "liked", "stars", "star", "review", "think", "thought", "lot", "way", "bit", "did",
    "does", "didn", "doesn", "don", "isn", "wasn", "ve", "ll", "time", "end", "ending", "new",
    "make", "makes", "made", "know", "going", "got", "get", "little", "want", "say", "said",
    "mystery", "mysteries", "thriller", "thrillers", "crime", "suspense",
}
vectorizer = CountVectorizer(
    stop_words=sorted(ENGLISH_STOP_WORDS | EXTRA_STOP_WORDS),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b",  # words of 3+ letters
    min_df=5, max_df=0.5, max_features=50_000, dtype=np.float32,
)
WORD_COUNTS = vectorizer.fit_transform(docs).tocsr()  # books x words
VOCAB = vectorizer.get_feature_names_out()
WORD_SHARES = normalize(WORD_COUNTS, norm="l1")       # each book's word counts as shares
print(f"{WORD_COUNTS.shape[1]:,} words in the vocabulary")

# Books x tags matrix (1 = the book carries the tag) and each tag's group (place, era, tone, ...)
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from subgenres import SUBGENRES

TAG_NAMES = list(SUBGENRES)
TAG_GROUP = np.array([SUBGENRES[t][0] for t in TAG_NAMES])
rows, cols = [], []
for r, book_tags in enumerate(tags["tags"].to_list()):
    for t in book_tags:
        rows.append(r)
        cols.append(TAG_NAMES.index(t))
TAGS = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(N, len(TAG_NAMES)))
TAG_BASE = np.asarray(TAGS.mean(axis=0)).ravel()  # share of all books carrying each tag

In [ ]:
def shuffle(labels):
    # The same cluster sizes, but books assigned at random: what "no signal" looks like.
    out = labels.copy()
    keep = np.flatnonzero(labels >= 0)
    out[keep] = np.random.default_rng(SEED).permutation(labels[keep])
    return out


def membership(labels):
    # (clusters x books) 0/1 matrix; noise (-1) is skipped.
    keep = labels >= 0
    clusters = np.unique(labels[keep])
    rows = np.searchsorted(clusters, labels[keep])
    M = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, np.flatnonzero(keep))), shape=(len(clusters), len(labels)))
    return clusters, M


def c_tf_idf(labels):
    # Returns the cluster ids, a (clusters x words) matrix of c-TF-IDF scores, and the idf per word.
    clusters, M = membership(labels)
    counts = M @ WORD_COUNTS                                # word counts per cluster
    tf = normalize(counts, norm="l1")                       # how common each word is in the cluster
    word_total = np.asarray(counts.sum(axis=0)).ravel()
    idf = np.log1p((counts.sum() / len(clusters)) / np.maximum(word_total, 1)).astype(np.float32)
    return clusters, tf.multiply(idf).tocsr(), idf


def top_words(scores, n=TOP_N_WORDS):
    return np.argsort(-scores.toarray(), axis=1)[:, :n]


def ctfidf_purity(labels):
    # Match every book's text to the closest cluster keyword profile: share matched to its own cluster.
    clusters, cluster_scores, idf = c_tf_idf(labels)
    in_cluster = labels >= 0
    book_profiles = normalize(WORD_SHARES[in_cluster].multiply(idf).tocsr())
    closest = clusters[(book_profiles @ normalize(cluster_scores).T).toarray().argmax(axis=1)]
    top = top_words(cluster_scores)
    return float((closest == labels[in_cluster]).mean()), float(len(np.unique(top)) / top.size)


def genre_purity(labels):
    # In each cluster, the share of its most common sub-genre (weighted by cluster size).
    use = (labels >= 0) & counts_for_purity
    return float(contingency_matrix(genre[use], labels[use]).max(axis=0).sum() / use.sum())


def tag_coverage(labels):
    # Share of each cluster's books that carry each tag: (clusters x tags).
    clusters, M = membership(labels)
    sizes = np.asarray(M.sum(axis=1)).ravel()
    return clusters, sizes, (M @ TAGS).toarray() / sizes[:, None]


def group_lift(labels, groups):
    # For each cluster: how much more common its best tag (from these groups) is than in all books.
    # Book-weighted average over clusters, minus the same for random clusters.
    idx = np.flatnonzero(np.isin(TAG_GROUP, groups))
    def gain(lab):
        _, sizes, cov = tag_coverage(lab)
        return float(((cov[:, idx] - TAG_BASE[idx]).max(axis=1) * sizes).sum() / sizes.sum())
    return gain(labels) - gain(shuffle(labels))


def matched_share(labels):
    # Share of books in clusters with one tag covering >= 40% of the cluster, >= 2x as common as usual.
    _, sizes, cov = tag_coverage(labels)
    matched = ((cov >= MATCH_COVERAGE) & (cov / np.maximum(TAG_BASE, 1e-9) >= MIN_LIFT)).any(axis=1)
    return float(sizes[matched].sum() / sizes.sum())


def score(labels, Z):
    # All the scores for one clustering. labels = cluster number per book (-1 = noise), Z = its UMAP vectors.
    in_cluster = labels >= 0
    k = len(np.unique(labels[in_cluster]))
    result = {"n_clusters": k, "noise_frac": float(1 - in_cluster.mean()),
              "max_cluster_share": float(np.bincount(labels[in_cluster]).max() / len(labels)) if k else 1.0}
    if k < 2:
        return result
    sample = None if in_cluster.sum() <= SILHOUETTE_SAMPLE else SILHOUETTE_SAMPLE
    result["silhouette"] = float(silhouette_score(Z[in_cluster], labels[in_cluster], sample_size=sample, random_state=SEED))
    result["genre_purity"] = genre_purity(labels)
    result["purity_lift"] = result["genre_purity"] - genre_purity(shuffle(labels))
    use = in_cluster & counts_for_purity
    result["nmi"] = float(normalized_mutual_info_score(genre[use], labels[use]))
    result["ctfidf_purity"], result["topic_diversity"] = ctfidf_purity(labels)
    result["ctfidf_lift"] = result["ctfidf_purity"] - ctfidf_purity(shuffle(labels))[0]
    result["premise_lift"] = group_lift(labels, PREMISE_GROUPS)
    result["tone_lift"] = group_lift(labels, ["tone"])
    result["matched_share"] = matched_share(labels)
    return result


def usable(row, natural=False):
    ok = (MIN_CLUSTERS <= row["n_clusters"] <= MAX_CLUSTERS and row["max_cluster_share"] <= MAX_CLUSTER_SHARE
          and row.get("silhouette", -1) >= MIN_SILHOUETTE)
    return ok and (not natural or row["noise_frac"] <= MAX_NATURAL_NOISE)


def add_rank_score(results):
    # Average percentile rank of the three scores (ranks, because the scores are on different scales).
    rank = pl.col("purity_lift").rank() + pl.col("ctfidf_lift").rank() + pl.col("premise_lift").rank()
    return results.with_columns((rank / (3 * pl.len())).alias("rank_score"))


def agreement(label_sets):
    # Average Adjusted Rand Index between every pair of clusterings (1 = identical, 0 = chance).
    pairs = [(a, b) for i, a in enumerate(label_sets) for b in label_sets[i + 1:]]
    return float(np.mean([adjusted_rand_score(a, b) for a, b in pairs]))


def assign_noise_to_nearest(labels, Z):
    # Every noise book (-1) goes to the cluster with the nearest centre, so every book has a cluster.
    out = labels.copy()
    noise = labels < 0
    clusters = np.unique(labels[~noise])
    if not noise.any() or clusters.size == 0:
        return out
    centres = np.vstack([Z[labels == c].mean(axis=0) for c in clusters])
    P = Z[noise]
    d = (P ** 2).sum(1)[:, None] - 2 * P @ centres.T + (centres ** 2).sum(1)[None, :]
    out[noise] = clusters[d.argmin(axis=1)]
    return out


SHOW = ["n_clusters", "noise_frac", "max_cluster_share", "silhouette", "purity_lift", "ctfidf_lift",
        "premise_lift", "tone_lift", "matched_share"]

## 4. K-Means: try different numbers of clusters

Every `k` is tried on every UMAP seed. `rank_score` is computed per seed, then averaged. The **seed noise** is
how much a k's `rank_score` moves between seeds (on average): every k within that of the best is a tie, and
the most stable of them wins.

In [ ]:
start = time.time()
rows, km_labels = [], {}
for k in KMEANS_K:
    for s in SEEDS:
        km_labels[(k, s)] = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X[s])
        rows.append({"k": k, "seed": s} | score(km_labels[(k, s)], X[s]))
kmeans_results = pl.DataFrame(rows)
print(f"Tried {len(KMEANS_K)} values of k on {len(SEEDS)} seeds in {(time.time() - start) / 60:.0f} min")

per_seed = pl.concat([add_rank_score(kmeans_results.filter(pl.col("seed") == s)) for s in SEEDS])
kmeans_mean = per_seed.group_by("k").agg(
    [pl.col(c).mean() for c in SHOW + ["rank_score"]]
    + [(pl.col("rank_score").max() - pl.col("rank_score").min()).alias("seed_range")]
).sort("k")
kmeans_mean = kmeans_mean.with_columns(
    pl.struct(kmeans_mean.columns).map_elements(usable, return_dtype=pl.Boolean).alias("usable"))
seed_noise = float(kmeans_mean["seed_range"].mean())
ranked = kmeans_mean.filter("usable").sort("rank_score", descending=True)
if ranked.is_empty():
    raise ValueError("No usable K-Means setting: check MIN_CLUSTERS, MAX_CLUSTER_SHARE and MIN_SILHOUETTE.")
shortlist = ranked.filter(pl.col("rank_score") >= ranked["rank_score"][0] - seed_noise)
if shortlist.height < 3:
    shortlist = ranked.head(3)

# Stability of each tie: agreement between seeds, and between all books and 90% of them
rng = np.random.default_rng(SEED)
stab = []
for k in shortlist["k"].to_list():
    between_seeds = agreement([km_labels[(k, s)] for s in SEEDS])
    subsample = []
    for _ in range(SUBSAMPLES):
        idx = np.sort(rng.choice(N, int(0.9 * N), replace=False))
        refit = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X[MAIN][idx])
        subsample.append(adjusted_rand_score(km_labels[(k, MAIN)][idx], refit))
    stab.append({"k": k, "between_seeds": between_seeds, "subsample": float(np.mean(subsample)),
                 "stability": (between_seeds + float(np.mean(subsample))) / 2})
shortlist = shortlist.join(pl.DataFrame(stab), on="k").sort("stability", descending=True)
best_k = int(shortlist["k"][0])
print(f"Seed noise of rank_score: {seed_noise:.3f}. Tied with the best: k = {sorted(shortlist['k'].to_list())}")
print(f"Most stable of them: k = {best_k}")
shortlist.select("k", "rank_score", "stability", "between_seeds", "subsample", *SHOW[3:]).with_columns(pl.selectors.float().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
r = kmeans_mean.sort("k")
for ax, col, title in zip(axes, ["purity_lift", "ctfidf_lift", "premise_lift"], ["Purity lift", "c-TF-IDF lift", "Premise lift"]):
    ax.plot(r["k"], r[col])
    ax.axvline(best_k, color="black", linestyle=":", linewidth=1)
    ax.set_title(title); ax.set_xlabel("number of clusters (k)")
plt.suptitle(f"K-Means scores, average of {len(SEEDS)} UMAP seeds (dotted line = chosen k = {best_k})", x=0.01, ha="left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "kmeans_scores.png", bbox_inches="tight")
plt.show()

## 5. HDBSCAN: try different settings

HDBSCAN is slow (about 45 s per setting), so every setting is tried on the main UMAP seed only. Each fit is
scored twice: **natural** (noise books stay unclustered; scored on the clustered books) and **forced** (every
noise book moved to the nearest cluster; scored on all books). The 5 best of each are then re-run on the other
seeds: a setting that fails the rules on any seed is dropped, and the one that agrees most across seeds wins.

In [ ]:
start = time.time()
rows, hdb_labels = [], {}
for mcs in HDBSCAN_MIN_CLUSTER_SIZE:
    for ms in HDBSCAN_MIN_SAMPLES:
        for method in HDBSCAN_METHODS:
            params = (mcs, ms, method)
            if max(mcs, ms or mcs) >= N // 2:  # too big for a small (quick-test) sample
                continue
            labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms, cluster_selection_method=method,
                             copy=True, n_jobs=-1).fit_predict(X[MAIN])
            hdb_labels[(params, MAIN)] = labels
            base = {"min_cluster_size": mcs, "min_samples": ms if ms is not None else mcs, "method": method, "params": str(params)}
            if MIN_CLUSTERS <= labels.max() + 1 <= MAX_CLUSTERS:  # outside the range it can't be usable: skip scoring
                rows.append(base | {"version": "natural"} | score(labels, X[MAIN]))
                rows.append(base | {"version": "forced", "natural_noise_frac": float(np.mean(labels < 0))}
                            | score(assign_noise_to_nearest(labels, X[MAIN]), X[MAIN]))
            else:
                rows += [base | {"version": v, "n_clusters": int(labels.max() + 1)} for v in ("natural", "forced")]
hdbscan_results = pl.DataFrame(rows, infer_schema_length=None)
print(f"Tried {len(hdb_labels)} settings in {(time.time() - start) / 60:.0f} min")

In [ ]:
start = time.time()
picks, checks = {}, []
for version in ["natural", "forced"]:
    natural = version == "natural"
    candidates = hdbscan_results.filter(pl.col("version") == version)
    if "silhouette" in candidates.columns:  # settings outside the cluster range were not scored
        candidates = candidates.filter(pl.col("silhouette").is_not_null())
        candidates = candidates.filter(pl.struct(candidates.columns).map_elements(lambda r: usable(r, natural), return_dtype=pl.Boolean))
    if "silhouette" not in candidates.columns or candidates.is_empty():
        raise ValueError(f"No usable HDBSCAN {version} setting. With a small sample of books (SAMPLE_BOOKS), "
                         "lower MIN_CLUSTERS in the settings, or use at least 3,000 books.")
    top = add_rank_score(candidates).sort("rank_score", descending=True).head(HDBSCAN_SHORTLIST)
    for r in top.iter_rows(named=True):
        params = ast.literal_eval(r["params"])  # back from text to (min_cluster_size, min_samples, method)
        labels = {}
        for s in SEEDS:
            if (params, s) not in hdb_labels:
                hdb_labels[(params, s)] = HDBSCAN(min_cluster_size=params[0], min_samples=params[1],
                                                  cluster_selection_method=params[2], copy=True, n_jobs=-1).fit_predict(X[s])
            labels[s] = hdb_labels[(params, s)] if natural else assign_noise_to_nearest(hdb_labels[(params, s)], X[s])
        other_seeds = [score(labels[s], X[s]) for s in SEEDS if s != MAIN]
        checks.append({"version": version, "params": r["params"], "rank_score": r["rank_score"],
                       "usable_on_every_seed": all(usable(o, natural) for o in other_seeds),
                       "between_seeds": agreement(list(labels.values())),
                       **{c: float(np.mean([r[c]] + [o.get(c, np.nan) for o in other_seeds]))
                          for c in ["n_clusters", "noise_frac", "purity_lift", "premise_lift"]},
                       "labels": labels})
    version_checks = sorted([c for c in checks if c["version"] == version],
                            key=lambda c: (c["usable_on_every_seed"], c["between_seeds"]), reverse=True)
    picks[version] = version_checks[0]
    print(f"HDBSCAN {version}: pick {picks[version]['params']} (agreement between seeds {picks[version]['between_seeds']:.3f})")
print(f"Re-ran the shortlists in {(time.time() - start) / 60:.0f} min")
hdbscan_checks = pl.DataFrame([{k: v for k, v in c.items() if k != "labels"} for c in checks])
hdbscan_checks.with_columns(pl.selectors.float().round(3))

## 6. The final models

Each picked setting was run on every UMAP seed. The seed whose clusters agree most with the other seeds is kept
(the most "typical" result, not a lucky or unlucky one).

In [ ]:
def most_typical_seed(labels_by_seed):
    agree = {s: np.mean([adjusted_rand_score(labels_by_seed[s], labels_by_seed[o]) for o in SEEDS if o != s]) for s in SEEDS}
    best = max(agree, key=agree.get)
    return best, float(agree[best])


final = {}
km_seed, km_agree = most_typical_seed({s: km_labels[(best_k, s)] for s in SEEDS})
final["kmeans"] = {"params": {"n_clusters": best_k}, "seed": km_seed, "agreement": km_agree, "labels": km_labels[(best_k, km_seed)]}
for version in ["natural", "forced"]:
    seed, agree = most_typical_seed(picks[version]["labels"])
    mcs, ms, method = ast.literal_eval(picks[version]["params"])
    final[f"hdbscan_{version}"] = {"params": {"min_cluster_size": mcs, "min_samples": ms, "cluster_selection_method": method},
                                   "seed": seed, "agreement": agree, "labels": picks[version]["labels"][seed]}
comparison = pl.DataFrame([{"model": name, "params": str(m["params"]), "umap_seed": m["seed"], "agreement": m["agreement"]}
                           | score(m["labels"], X[m["seed"]]) for name, m in final.items()])
comparison.select("model", "params", "umap_seed", "agreement", *SHOW).with_columns(pl.selectors.float().round(3))

## 7. What is each cluster about?

Each cluster is described by its **tags** (sub-genres much more common in it than overall: coverage = share of
its books with the tag, ×lift = how many times more common than in all books), its **top words** (c-TF-IDF)
and its **best-known books**. "unmatched" = no single tag covers at least 40% of the cluster.

In [ ]:
ratings = books["ratings_count"].to_numpy()


def describe_clusters(labels, model_name, n_tags=4):
    clusters, sizes, cov = tag_coverage(labels)
    lift = cov / np.maximum(TAG_BASE, 1e-9)
    _, cluster_scores, _ = c_tf_idf(labels)
    words = top_words(cluster_scores)
    rows = []
    for i, c in enumerate(clusters):
        strength = np.where(lift[i] >= MIN_LIFT, cov[i], -1)  # enriched tags, most common first
        top = [j for j in np.argsort(-strength)[:n_tags] if strength[j] > 0]
        members = np.flatnonzero(labels == c)
        best_known = members[np.argsort(-ratings[members])[:4]]
        rows.append({
            "model": model_name, "cluster": int(c), "size": int(sizes[i]),
            "name": " + ".join(TAG_NAMES[j] for j in top[:2]) if top else "(no clear tag)",
            "matched": bool(top) and cov[i, top[0]] >= MATCH_COVERAGE,
            "tags": ", ".join(f"{TAG_NAMES[j]} {cov[i, j]:.0%} (x{lift[i, j]:.1f})" for j in top),
            "top_words": ", ".join(VOCAB[words[i]]),
            "best_known": " | ".join(t[:45] for t in titles[best_known]),
        })
    return pl.DataFrame(rows).sort("size", descending=True)


topics = {name: describe_clusters(m["labels"], name) for name, m in final.items()}
print(f"K-Means: {topics['kmeans'].height} clusters")
topics["kmeans"].drop("model")

In [ ]:
m = final["hdbscan_natural"]
print(f"HDBSCAN natural: {topics['hdbscan_natural'].height} clusters, plus {np.mean(m['labels'] < 0):.0%} of books as noise")
topics["hdbscan_natural"].drop("model")

In [ ]:
print(f"HDBSCAN forced: {topics['hdbscan_forced'].height} clusters, every book assigned")
topics["hdbscan_forced"].drop("model")

**How well does each book fit its K-Means cluster?** `kmeans_fit` = distance to its own cluster centre ÷ distance
to the second-nearest centre. Small = clearly in its cluster; close to 1 = halfway between two clusters.

In [ ]:
Z = X[final["kmeans"]["seed"]]
km = final["kmeans"]["labels"]
centres = np.vstack([Z[km == c].mean(axis=0) for c in range(best_k)])
dist = np.sqrt(np.maximum((Z ** 2).sum(1)[:, None] - 2 * Z @ centres.T + (centres ** 2).sum(1)[None, :], 0))
nearest_two = np.sort(dist, axis=1)[:, :2]
kmeans_fit = nearest_two[:, 0] / np.maximum(nearest_two[:, 1], 1e-9)
print(f"Median kmeans_fit {np.median(kmeans_fit):.2f}; books almost between two clusters (> 0.9): {np.mean(kmeans_fit > 0.9):.1%}")

## 8. Cluster map

In [ ]:
# A 2-D picture of the books (t-SNE), for looking only; the models don't use it.
# Grey = all books, color = the 3 largest clusters. Large datasets are shown as a random sample.
show = np.sort(np.random.default_rng(SEED).choice(N, TSNE_MAX_POINTS, replace=False)) if N > TSNE_MAX_POINTS else np.arange(N)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, name, title in [(axes[0], "kmeans", f"K-Means (k={best_k})"), (axes[1], "hdbscan_natural", "HDBSCAN natural")]:
    m = final[name]
    XY = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(X[m["seed"]][show])
    labels = m["labels"][show]
    ax.scatter(*XY.T, s=3, color="lightgrey")
    for row, color in zip(topics[name].head(3).iter_rows(named=True), ["tab:blue", "tab:orange", "tab:green"]):
        inside = labels == row["cluster"]
        ax.scatter(*XY[inside].T, s=6, color=color, label=f"#{row['cluster']} ({row['size']}): {row['name']}")
    ax.set_title(title, loc="left"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "cluster_map.png", bbox_inches="tight")
plt.show()

## 9. Save the results

In [ ]:
books.select("book_id", "title", "genre_label").with_columns(
    pl.Series("primary_genre", genre),
    pl.Series("kmeans_cluster", final["kmeans"]["labels"].astype(np.int32)),
    pl.Series("kmeans_fit", kmeans_fit.astype(np.float32)),                                 # close to 1 = borderline
    pl.Series("hdbscan_cluster", final["hdbscan_natural"]["labels"].astype(np.int32)),     # -1 = noise
    pl.Series("hdbscan_cluster_all", final["hdbscan_forced"]["labels"].astype(np.int32)),  # forced version: every book
).write_parquet(OUTPUT_DIR / "cluster_assignments.parquet")
pl.concat(list(topics.values())).write_parquet(OUTPUT_DIR / "cluster_topics.parquet")
kmeans_results.write_parquet(OUTPUT_DIR / "tuning_kmeans.parquet")
hdbscan_results.write_parquet(OUTPUT_DIR / "tuning_hdbscan.parquet")
kmeans_model = KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit(X[final["kmeans"]["seed"]])
joblib.dump(kmeans_model, OUTPUT_DIR / "kmeans_model.joblib")  # can place new books with .predict()

summary = {
    "inputs": {"n_books": N, "dims": X[MAIN].shape[1], "umap_seeds": SEEDS},
    "kmeans_seed_noise": seed_noise,
    "models": {name: {"params": m["params"], "umap_seed": m["seed"], "agreement_between_seeds": m["agreement"]}
               for name, m in final.items()},
    "new_books": f"reduce with clustering_inputs/umap_model_seed{final['kmeans']['seed']}.joblib, then kmeans_model.predict()",
    "comparison": comparison.to_dicts(),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("Saved to", OUTPUT_DIR)